# Fetch best Milo result

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # miloR
            library(miloR), 
            library(ggbeeswarm), 
            
            # SingleCellExperiment
            library(SingleCellExperiment), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # Plotting 
            library(patchwork), 
            library(cowplot), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv="p.3.10.16-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

## Import Milo results 

In [ ]:
res_0 <- readRDS("data/milo/resl_0.rds")[[3]]
res_1 <- readRDS("data/milo/resl_1.rds")[[3]]
res_2 <- readRDS("data/milo/resl_2.rds")[[3]]

In [ ]:
dabp <- function(res, alpha=0.05, group_by="celltype_low", y_breaks=c(-6, -3, 0, +3, +6)) {
    
    # Beeswarm 
    data_1 <- suppressMessages(plotDAbeeswarm(res[[2]][[1]], group.by=group_by, alpha=1.0))$data %>% dplyr::mutate(group="Tx")
    data_2 <- suppressMessages(plotDAbeeswarm(res[[2]][[2]], group.by=group_by, alpha=1.0))$data %>% dplyr::mutate(group="D14")
    data_3 <- suppressMessages(plotDAbeeswarm(res[[2]][[3]], group.by=group_by, alpha=1.0))$data %>% dplyr::mutate(group="D100") 
    data_4 <- suppressMessages(plotDAbeeswarm(res[[2]][[4]], group.by=group_by, alpha=1.0))$data %>% dplyr::mutate(group="GVHD")
                               
    data <- rbind(data_1, data_2, data_3, data_4)
    data$group <- factor(data$group, levels=c("Tx", "D14", "D100", "GVHD"))
    
    data <- data %>%
        mutate(is_signif = ifelse(SpatialFDR < alpha, 1, 0)) %>%
        mutate(logFC_color = ifelse(is_signif==1, logFC, NA)) %>%
        arrange(group_by) %>%
        mutate(Nhood=factor(Nhood, levels=unique(Nhood))) %>%
        mutate(pos_x = pos_x, pos_y=pos_y) 

    if(is.null(y_breaks)) {

        y_breaks <- c(-floor(max(abs(data$logFC))), -floor(max(abs(data$logFC)))/2, 0, floor(max(abs(data$logFC)))/2, floor(max(abs(data$logFC))))
        
    }
    
    n_groups <- length(unique(data$group_by))

    # Make plot for each data entry 
    data <- split(data, f=data$group)

    dabp <- lapply(names(data), function(i) {

        x <- data[[i]]
        
        # Plot without color gradient to avoid NA issue
        dabp <- ggplot(x, aes(pos_x, pos_y, color=logFC_color)) +
            guides(color="none") +
            xlab("") + ylab("Log Fold Change") +
            scale_x_discrete(labels=parse(text=setNames(levels(x$group_by), seq(1,n_groups)))) + 
            
            geom_hline(yintercept=0, size=0.5) + 
            geom_point(size=0.1) + 
            stat_summary(aes(x=group_by, y=logFC), fun.y=median, geom="point", size=1, color="black") +
            coord_flip() +
            scale_y_continuous(breaks=y_breaks, labels=y_breaks, limits = c(-6.5, +6.5)) +
            theme(axis.text.y=element_text(vjust=0.5, hjust=1)) + 
            theme_global_set(4)

        color_check <- all(is.na(x$logFC_color))

        if(color_check) {
    
            dabp <- dabp
            
        } else {
    
            dabp <-dabp + scale_color_gradient2(low=color$time_point[1], high=color$time_point[i])
            
        }

        return(dabp)
        
    }
                  )


    # Set plotting size 
    dabp <- lapply(dabp, function(x) {

        x %>% egg::set_panel_size(., width=unit(1.5, "cm"), height=unit(n_groups*0.25, "cm"))
        
    }
          )

                               
    return(dabp)
}


## Blood

In [ ]:
dabp_0 <- dabp(res_0)

In [ ]:
pdf("result/figures/figure_6/milo_blood_time_points.pdf", width=10, height=5)

gridExtra::grid.arrange(

    dabp_0[[1]], 
    dabp_0[[2]], 
    dabp_0[[3]], 
    dabp_0[[4]], ncol=4, nrow=1

) 

dev.off()

## Skin hematopoietic 

In [ ]:
dabp_1 <- dabp(res_1)

In [ ]:
pdf("result/figures/figure_6/milo_skin_hematopoietic_time_points.pdf", width=5, height=5)

gridExtra::grid.arrange(

    dabp_1[[1]], 
    dabp_1[[2]], 
    dabp_1[[3]], 
    dabp_1[[4]], ncol=4, nrow=1

) 

dev.off()

## Skin non-hematopoietic

In [ ]:
dabp_2 <- dabp(res_2)

In [ ]:
pdf("result/figures/figure_6/milo_skin_non_hematopoietic_time_points.pdf", width=5, height=5)

gridExtra::grid.arrange(

    dabp_2[[1]], 
    dabp_2[[2]], 
    dabp_2[[3]], 
    dabp_2[[4]], ncol=4, nrow=1

) 

dev.off()

# Session info

In [ ]:
sessionInfo()